In [55]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import ast
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA, NMF
from sklearn.manifold import TSNE, Isomap, MDS
import umap
import matplotlib.patches as mpatches
from scipy.stats import entropy

In [56]:
PROJECT_ROOT = Path.cwd().parent

DATA_RAW = PROJECT_ROOT / "data" / "raw"

DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

DATA_IMAGE = PROJECT_ROOT / "data" / "images"
DATA_IMAGE.mkdir(parents=True, exist_ok=True)

print("Raw data folder:", DATA_RAW)
print("Processed data folder:", DATA_PROCESSED)


Raw data folder: c:\Users\ccana\Documents\Doutorado\VISEMTracking\data\raw
Processed data folder: c:\Users\ccana\Documents\Doutorado\VISEMTracking\data\processed


In [57]:
df = pd.read_csv(DATA_RAW / "visem_all_trajectory.csv")

num_traj = df["trajectory_id"].nunique()
num_traj_0 = df[df["ftid"]==0]["trajectory_id"].nunique()

print(f"Quantidade de trajectory_id únicos (ft_id com 0, 1 e 2)): {num_traj}")
print(f"Quantidade de trajectory_id únicos (ft_id com 0)): {num_traj_0}")

df 

Quantidade de trajectory_id únicos (ft_id com 0, 1 e 2)): 1176
Quantidade de trajectory_id únicos (ft_id com 0)): 1121


,user_id,timestamp,trajectory_id,ftid,x,y,w,h
0,11,0,ckz3v9nzv00033867jsekqdcl,0,0.275781,0.412500,0.026562,0.037500
1,11,0,ckz6ru2eb0001386lefk6js4m,0,0.238281,0.583333,0.026562,0.037500
2,11,0,ckz6ru3um0003386l39xwv0yk,0,0.246094,0.847917,0.026562,0.037500
3,11,0,ckz6ru5610005386l8uoys4lc,0,0.194531,0.845833,0.026562,0.037500
4,11,0,ckz6ru6of0007386lhf2t4dtw,0,0.083594,0.806250,0.026562,0.037500
...,...,...,...,...,...,...,...,...
656330,82,9,cl3g5f3gs0007396c1pgai157,0,0.674219,0.165546,0.029687,0.039583
656331,82,9,cl3g5fe45000b396c31bq1d83,0,0.684229,0.159570,0.012646,0.031445
656332,82,9,cl3g64xob000j396cq58md4lq,0,0.872559,0.385682,0.032813,0.043750
656333,82,9,cl4z9vsrf00033y6fzi92k4b3,2,0.964844,0.816667,0.032813,0.045833


In [58]:
df_symbol = (
    df
    .sort_values("timestamp")  # garante ordem temporal
    .groupby("trajectory_id", as_index=False)
    .agg(
        ftid=("ftid", "first"),
        user_id=("user_id", "first"),
        frame_inicial=("timestamp", "min"),
        frame_final=("timestamp", "max"),
        trajectory_xy=("x", lambda s: list(zip(s, df.loc[s.index, "y"])))
    )
)

df_symbol


,trajectory_id,ftid,user_id,frame_inicial,frame_final,trajectory_xy
0,ckyw6zzlj001r3867thf0fuy7,0,14,0,748,"[(0.20859375, 0.825), (0.20859375, 0.814583333..."
1,ckyw704kw001v3867kvyjtx6k,0,14,0,748,"[(0.79609375, 0.7979166666666667), (0.7640625,..."
2,ckyw708pn001z386779fr849h,0,14,0,748,"[(0.82734375, 0.1239583333333333), (0.82741168..."
3,ckyw723jy00233867gakwo9e7,0,14,15,748,"[(0.85078125, 0.0208333333333333), (0.85546875..."
4,ckyw7dzmx00273867fyj9lsnv,0,14,749,1469,"[(0.0796875, 0.3875), (0.0830078125, 0.3859375..."
...,...,...,...,...,...,...
1171,cl696k4y4000b3b6gofhyu663,1,54,997,1469,"[(0.38046875, 0.840625), (0.38046875, 0.840625..."
1172,cl696kcfx000f3b6gr3k0oqk4,1,54,997,1469,"[(0.50859375, 0.7895833333333333), (0.50859375..."
1173,cl696kim5000j3b6g3waaz3ij,0,54,997,1469,"[(0.47734375, 0.8979166666666667), (0.47734375..."
1174,cl696ldsk000n3b6g3cevz2tg,0,54,997,1469,"[(0.68828125, 0.2802083333333333), (0.68828125..."


In [64]:
df2 = pd.read_csv('symbolic.csv')
df_symbol['trajectory_id'] = df_symbol['trajectory_id'].map(
    df2.set_index('trajectory_id')['trajectory_label']
)

# ftid = 1 → cluster
mask1 = df_symbol['ftid'] == 1
df_symbol.loc[mask1, 'trajectory_id'] = [
    f"cluster_{i}" for i in range(1, mask1.sum() + 1)
]

# ftid = 2 → pin
mask2 = df_symbol['ftid'] == 2
df_symbol.loc[mask2, 'trajectory_id'] = [
    f"pin_{i}" for i in range(1, mask2.sum() + 1)
]

df_symbol[df_symbol['ftid'].isin([1, 2])]

,trajectory_id,ftid,user_id,frame_inicial,frame_final,trajectory_xy
168,pin_1,2,60,0,498,"[(0.43359375, 0.2010416666666666), (0.43341686..."
281,pin_2,2,60,0,498,"[(0.36328125, 0.4520833333333333), (0.36315456..."
341,pin_3,2,54,0,996,"[(0.03125, 0.7854166666666667), (0.04375, 0.79..."
394,pin_4,2,47,0,738,"[(0.24375, 0.896875), (0.2484375, 0.8989583333..."
510,pin_5,2,19,471,643,"[(0.1671875, 0.4791666666666667), (0.166741071..."
532,pin_6,2,21,0,250,"[(0.3, 0.1041666666666666), (0.290625, 0.10416..."
561,pin_7,2,21,270,749,"[(0.084375, 0.2666666666666666), (0.08671875, ..."
590,pin_8,2,21,753,1469,"[(0.75390625, 0.4802083333333333), (0.75390625..."
613,pin_9,2,35,0,1160,"[(0.67421875, 0.1927083333333333), (0.69921875..."
770,pin_10,2,12,132,234,"[(0.68828125, 0.0364583333333333), (0.68671875..."


In [60]:
df2

,Unnamed: 0,trajectory_id,user_id,frame_inicial,frame_final,symbolic_movement_10,symbolic_movement_15,symbolic_movement_20,movement_list,cluster,...,mds_1,mds_2,nmf_1,nmf_2,umap_1,umap_2,trajectory_xy,trajectory_xy_translate,trajectory_xy_rotated,trajectory_label
0,0,ckyw6zzlj001r3867thf0fuy7,14,0,748,"['Muito_Rapido_Leste', 'Muito_Rapido_Leste', '...","['Muito_Rapido_Leste', 'Muito_Rapido_Norte', '...","['Muito_Rapido_Leste', 'Muito_Rapido_Norte', '...","['Muito_Rapido_Leste', 'Muito_Rapido_Leste', '...",2,...,1.303985,-0.527354,0.205622,0.000000,-4.392757,11.343345,"[(0.20859375, 0.825), (0.16640625, 0.655833333...","[(0.0, 0.0), (-0.0421874999999999, -0.16916666...","[(0.0, 0.0), (0.1652272724064276, -0.055651547...",traj_1
1,1,ckyw704kw001v3867kvyjtx6k,14,0,748,"['Muito_Rapido_Oeste', 'Muito_Rapido_Oeste', '...","['Muito_Rapido_Oeste', 'Muito_Rapido_Sul', 'Mu...","['Muito_Rapido_Oeste', 'Muito_Rapido_Sul', 'Mu...","['Muito_Rapido_Oeste', 'Muito_Rapido_Oeste', '...",2,...,1.092192,-0.389062,0.191151,0.000000,-4.158039,11.112620,"[(0.79609375, 0.7979166666666667), (0.73203125...","[(0.0, 0.0), (-0.0640625, -0.0639880952380953)...","[(0.0, 0.0), (-0.0879889545935215, 0.021364084...",traj_2
2,2,ckyw708pn001z386779fr849h,14,0,748,"['Rapido_Leste', 'Rapido_Leste', 'Rapido_Leste...","['Rapido_Leste', 'Rapido_Leste', 'Rapido_Leste...","['Rapido_Leste', 'Rapido_Leste', 'Rapido_Oeste...","['Rapido_Leste', 'Rapido_Leste', 'Rapido_Leste...",0,...,-0.967089,-2.095728,0.001224,0.132351,-0.246003,1.781761,"[(0.82734375, 0.1239583333333333), (0.82802309...","[(0.0, 0.0), (0.0006793478260869, 0.0009057971...","[(0.0, -0.0), (0.0008644725067896, -0.00073121...",traj_3
3,3,ckyw7dzmx00273867fyj9lsnv,14,749,1469,"['Muito_Rapido_Norte', 'Muito_Rapido_Norte', '...","['Muito_Rapido_Norte', 'Muito_Rapido_Norte', '...","['Muito_Rapido_Norte', 'Muito_Rapido_Oeste', '...","['Muito_Rapido_Norte', 'Muito_Rapido_Norte', '...",2,...,1.274650,-0.490624,0.204376,0.000000,-4.307722,11.328615,"[(0.0830078125, 0.3859375), (0.116796875, 0.37...","[(0.0, 0.0), (0.0337890624999999, -0.009374999...","[(0.0, 0.0), (0.0117804159580207, 0.0330274608...",traj_4
4,4,ckyw7e4o6002b3867ibaukox2,14,749,1469,"['Rapido_Norte', 'Rapido_Leste', 'Rapido_Leste...","['Rapido_Norte', 'Rapido_Leste', 'Rapido_Leste...","['Rapido_Leste', 'Rapido_Leste', 'Rapido_Leste...","['Rapido_Norte', 'Rapido_Leste', 'Rapido_Leste...",3,...,-1.273212,1.538022,0.014082,0.173779,-1.534151,4.334890,"[(0.355078125, 0.6385416666666667), (0.3523437...","[(0.0, 0.0), (-0.002734375, 0.000328947368421)...","[(0.0, 0.0), (-0.000283150812109, 0.0027394960...",traj_5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
802,802,cl691zdqu000h3b6gpesz4820,54,0,996,"['Medio_Leste', 'Medio_Leste', 'Medio_Leste', ...","['Medio_Leste', 'Medio_Leste', 'Medio_Leste', ...","['Medio_Leste', 'Medio_Leste', 'Medio_Leste', ...","['Medio_Leste', 'Medio_Leste', 'Medio_Leste', ...",4,...,-1.148223,-0.192960,0.000000,0.137068,1.195871,4.820254,"[(0.78984375, 0.909375), (0.7900937499999999, ...","[(0.0, 0.0), (0.0002499999999999, -0.00025), (...","[(0.0, 0.0), (0.0003116544421821, -0.000166947...",traj_803
803,803,cl696haqz00073b6gf0un301j,54,997,1469,"['Muito_Lento_Leste', 'Muito_Lento_Leste', 'Mu...","['Muito_Lento_Leste', 'Muito_Lento_Leste', 'Mu...","['Muito_Lento_Leste', 'Muito_Lento_Leste', 'Mu...","['Muito_Lento_Leste', 'Muito_Lento_Leste', 'Mu...",4,...,-0.248310,-0.139133,0.000000,0.060468,60.349365,8.808309,"[(0.128125, 0.5302083333333333), (0.128125, 0....","[(0.0, 0.0), (0.0, 0.0), (0.0, 0.0), (0.0, 0.0...","[(0.0, 0.0), (0.0, 0.0), (0.0, 0.0), (0.0, 0.0...",traj_804
804,804,cl696kim5000j3b6g3waaz3ij,54,997,1469,"['Muito_Lento_Leste', 'Muito_Lento_Leste', 'Mu...","['Muito_Lento_Leste', 'Muito_Lento_Leste', 'Mu...","['Muito_Lento_Leste', 'Muito_Lento_Leste', 'Mu...","['Muito_Lento_Leste', 'Muito_Lento_Leste', 'Mu...",4,...,-0.248310,-0.139133,0.000000,0.060468,60.493336,8.627234,"

In [63]:
import cv2
import os

user_ids = [12,13,14,15,19,21,22,23,24,29,30,35,36,38,47,52,54,60,82]

EXPECTED_FRAMES = 1470

for uid in user_ids:
    VIDEO_PATH = f"{uid}.mp4"
    OUTPUT_DIR = str(uid)

    os.makedirs(OUTPUT_DIR, exist_ok=True)

    cap = cv2.VideoCapture(VIDEO_PATH)

    if not cap.isOpened():
        raise RuntimeError(f"Não foi possível abrir o vídeo: {VIDEO_PATH}")

    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS)

    print(f"\nUsuário {uid}")
    print(f"FPS do vídeo: {fps}")
    print(f"Frames reportados: {frame_count}")

    frame_idx = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        cv2.imwrite(
            os.path.join(OUTPUT_DIR, f"{frame_idx:04d}.png"),
            frame
        )
        frame_idx += 1

    cap.release()

    print(f"Frames extraídos: {frame_idx}")

    if frame_idx != EXPECTED_FRAMES:
        print("⚠️ ATENÇÃO: número de frames diferente do esperado!")
    else:
        print("✅ Extração correta: 1470 frames")



Usuário 12
FPS do vídeo: 49.0
Frames reportados: 1470
Frames extraídos: 1470
✅ Extração correta: 1470 frames

Usuário 13
FPS do vídeo: 49.0
Frames reportados: 1470
Frames extraídos: 1470
✅ Extração correta: 1470 frames

Usuário 14
FPS do vídeo: 49.0
Frames reportados: 1470
Frames extraídos: 1470
✅ Extração correta: 1470 frames

Usuário 15
FPS do vídeo: 49.0
Frames reportados: 1470
Frames extraídos: 1470
✅ Extração correta: 1470 frames

Usuário 19
FPS do vídeo: 49.0
Frames reportados: 1470
Frames extraídos: 1470
✅ Extração correta: 1470 frames

Usuário 21
FPS do vídeo: 49.0
Frames reportados: 1470
Frames extraídos: 1470
✅ Extração correta: 1470 frames

Usuário 22
FPS do vídeo: 49.0
Frames reportados: 1470
Frames extraídos: 1470
✅ Extração correta: 1470 frames

Usuário 23
FPS do vídeo: 49.0
Frames reportados: 1470
Frames extraídos: 1470
✅ Extração correta: 1470 frames

Usuário 24
FPS do vídeo: 49.0
Frames reportados: 1470
Frames extraídos: 1470
✅ Extração correta: 1470 frames

Usuário 2